# Octahedral Patchy Particles

We write a basic implementation of octahedral patchy particles with colored interactions to encode for the octahedral DNA self-assembly of particles. The particles are composed of two particle types: **central particles** (which are the DNA origami frame, essentially) and **patches** (which are the particles representing the ssDNA strands emitted at the vertices).

In [1]:
%pip install jax-md

Note: you may need to restart the kernel to use updated packages.


In [1]:
import jax.numpy as jnp
from jax import config
import jax
config.update("jax_enable_x64", True) # necessary for 64-bit precision

from jax import jit, random, grad, value_and_grad, remat, jacfwd, vmap, lax
from jax.example_libraries import optimizers
from jax_md import space, smap, energy, minimize, quantity, simulate, partition, rigid_body, util
# from jax_md.colab_tools import renderer
from jax_md import partition

import numpy as np

## Construction of octahedron

Let us construct a `rigid_body.point_union_shape` to represent the octahedral patchy particle, then visualize it. For the purposes of this demo, I will assign four particles on the equatorial ring (arbitrarily defined) with a species A and the remaining two particles on the poles with a species B.

In [ ]:
# Constructs a single octahedral patchy particle as a RigidPointUnion — a rigid body
# described as a fixed union of point masses in body-frame coordinates.
#
# Each particle has 7 sites:
#   - 1 central site  (species 0): represents the DNA origami frame body
#   - 6 patch sites   (species 1): represent the 6 ssDNA sticky-end vertices
#     placed at ±RADIUS along each axis of the octahedron
#
# The energy functions later use these species labels to assign different interaction
# potentials to each type (LJ repulsion for centers, Morse attraction for patches).
def octahedron_shape(radius=0.5):
    particle_positions = jnp.array([
        [0.0,    0.0,    0.0],    # central site (origin of body frame)
        [radius, 0.0,    0.0],    # +x patch
        [-radius,0.0,    0.0],    # -x patch
        [0.0,    radius, 0.0],    # +y patch
        [0.0,   -radius, 0.0],    # -y patch
        [0.0,    0.0,    radius], # +z patch
        [0.0,    0.0,   -radius], # -z patch
    ], dtype=jnp.float64)

    # Species labels used to select rows/columns of the interaction epsilon matrices.
    # 0 = central site, 1 = patch site (all six patches treated as the same species here).
    particle_species = jnp.array([0, 1, 1, 1, 1, 1, 1], dtype=jnp.int32)

    # Patch masses are set to ~1e-8 (much smaller than the central mass of 1.0).
    # This keeps the centre of mass and principal axes essentially at the central site,
    # avoiding near-singular force gradients when patches overlap.
    # NOTE: this makes the moment of inertia tiny (~I ~ 2e-7), so the Langevin
    # integrator overdamps rotations; orientational DOF won't thermalize correctly
    # at the chosen dt. The translational dynamics are unaffected.
    patch_masses = jnp.linspace(0.1, 1.0, 6) * 1e-8
    masses = jnp.concatenate((jnp.array([1.0]), patch_masses), axis=0)

    # point_union_shape bundles the body-frame positions and masses into the
    # RigidPointUnion dataclass that JAX-MD's rigid-body integrator expects.
    shape = rigid_body.point_union_shape(particle_positions, masses).set(point_species=particle_species)
    return shape


Let us quickly visualize our octahedron, and make sure it looks fine.

In [3]:
RADIUS = 2.5
oct_shape = octahedron_shape(radius=RADIUS)

# arbitrary initial orientation. last two values must be around this for some reason
oct_orientation = rigid_body.Quaternion(jnp.array([[ 0, 0,  6e-01,  8e-01],]))
oct_rigid_body = rigid_body.RigidBody(center=jnp.array([[10.0, 10.0, 10.0]]), orientation=oct_orientation)
oct_position = vmap(rigid_body.transform, (0, None))(oct_rigid_body, oct_shape).reshape(-1, 3)

In [174]:
# # COLAB ONLY
# species = jnp.array([0, 1, 1, 1, 1, 2, 2], dtype=jnp.int32)
# diameters = jnp.where(species == 0, 1.0, 0.2) * (RADIUS * 2)

# renderer.render(20.0,
#                 {
#                     'particle': renderer.Sphere(oct_position, diameter=diameters),
#                 },
#                 resolution=(512, 512))

## Set up the simulation

In [ ]:
# ── System size ──────────────────────────────────────────────────────────────
# number_of_particles: how many octahedral particles to simulate.
#   Increase to 100–200 for production runs; 5 is convenient for quick tests.
# volume_density: packing fraction φ = N * V_particle / V_box.
#   Typical colloidal experiments: 0.01–0.05.
number_of_particles = 5
volume_density = 0.03

# Derive a cubic periodic box whose volume gives the target packing fraction.
# V_particle = (4/3)π r³ with r = RADIUS.
get_box_size = lambda phi, N, rad: (N * jnp.pi * 4 * rad**3 / phi / 3.0) ** (1/3)
box_size = get_box_size(volume_density, number_of_particles, RADIUS)
print("Box Size: {:.3f}".format(box_size))

# space.periodic sets up minimum-image periodic boundary conditions.
# disp_fn(a, b) returns the shortest displacement vector from b to a,
# and shift_fn(r, dr) moves a position by dr and wraps back into the box.
disp_fn, shift_fn = space.periodic(box_size)

# ── Integrator parameters ─────────────────────────────────────────────────────
dt = 1e-3          # time step (in reduced units); reduce if simulation is unstable
num_steps = int(3e5)  # total number of MD steps; 3×10^5 * dt = 300 time units
max_time  = num_steps * dt

# kT: thermal energy.  A function of time so it can be changed to an annealing
# schedule later, e.g.: kT = lambda t: 1.0 + (0.1 - 1.0) * (t / max_time)
# At kT ≈ D0 particles barely bind; kT << D0 promotes assembly.
kT = lambda t: 0.1

# gamma: friction coefficient for the Langevin thermostat.
# Higher gamma → stronger coupling to the heat bath, faster temperature equilibration
# but also more viscous (slower diffusion).
# RigidBody wraps separate translational (gamma) and rotational (3*gamma) coefficients.
gamma = 5.0
gamma = rigid_body.RigidBody(jnp.array([gamma]), jnp.array([3.0 * gamma]))

# save_every: record one trajectory frame every this many MD steps.
# 1000 steps * dt = 1 time unit between frames.
save_every = int(1e3)

# ── Interaction parameters ────────────────────────────────────────────────────
# D0: depth of the Morse potential well between patches (units of kT).
#   D0 >> kT → strong, essentially irreversible binding.
#   D0 ~ kT  → reversible assembly / annealing.
D0    = 10.0
ALPHA = 250.0   # soft-sphere steepness (only used in the initial minimisation)

key = random.PRNGKey(0)
print("=== Finished setup ===")


Now we need to initialize the positions and orientations of our patchy particles and make sure everything is good. Also, I'm going to initially separate out the positions using `soft_sphere` potentials to make sure each particle is not intersecting with another initially.

In [ ]:
# Place particles at random starting positions, then run energy minimisation to
# push them apart so no two particles overlap before the dynamics begin.
# Overlapping particles would create enormous repulsive forces that destabilise
# the integrator immediately.
central_particle_positions = random.uniform(key, (number_of_particles, 3),
                                            minval=0.0, maxval=box_size)

# soft_sphere_pair: purely repulsive potential U ∝ (σ/r)^α that vanishes
# when particles are separated.  We use σ = 2*RADIUS so each pair is separated
# to at least one particle diameter before the main simulation.
energy_fn = energy.soft_sphere_pair(disp_fn, sigma=RADIUS * 2)

# FIRE (Fast Inertial Relaxation Engine) is a gradient-descent minimiser
# well-suited to particle packings.  We run 5000 steps, which is usually
# enough to remove all overlaps.
init_fn, apply_fn = minimize.fire_descent(energy_fn, shift_fn)
state = init_fn(central_particle_positions)
apply_fn_jit = jit(lambda i, state: apply_fn(state))
state = lax.fori_loop(0, 5000, apply_fn_jit, state)

# The minimised positions become the initial condition for the dynamics.
central_particle_positions = state.position


In [ ]:
# Build the initial RigidBody state for the whole system.
#
# RigidBody is JAX-MD's representation of a collection of rigid bodies.
# It carries two fields:
#   .center      — centre-of-mass positions, shape (N, 3)
#   .orientation — unit quaternions encoding each body's rotation, shape (N, 4)
#                  Quaternion (w, x, y, z): [0, 0, 0.6, 0.8] is a ~90° rotation
#                  about the z-axis and is used here as an arbitrary common start.
#
# All N particles start with the same orientation; they will randomise quickly
# once the dynamics begin because the rotational friction is very high.
system_orientation = rigid_body.Quaternion(
    jnp.array([[0, 0, 6e-01, 8e-01] for _ in range(number_of_particles)])
)
system_rigid_body = rigid_body.RigidBody(
    center=central_particle_positions,
    orientation=system_orientation,
)

# Transform from body-frame (RigidBody) to world-frame flat positions for visualisation.
# vmap applies rigid_body.transform over the N particles in batch; the result is
# reshaped to (N*7, 3): 7 sites per particle laid out in particle order.
system_positions = vmap(rigid_body.transform, (0, None))(system_rigid_body, oct_shape).reshape(-1, 3)


In [ ]:
# Build per-site arrays for downstream visualisation and file output.
# Each octahedral particle contributes 7 sites (1 center + 6 patches),
# so for N particles there are N*7 sites total.
#
# species: integer label (0 = center, 1 = patch) for every site in
#   particle order [p0_center, p0_patch×6, p1_center, p1_patch×6, ...].
#   Reused when writing trajectory files and when assigning render radii.
#
# diameters: visual diameter for each site — 1.0 * (2*RADIUS) for center
#   spheres, 0.2 * (2*RADIUS) for the smaller patch spheres.
species   = jnp.array(list(oct_shape.point_species) * number_of_particles).flatten()
diameters = jnp.where(species == 0, 1.0, 0.2) * (RADIUS * 2)

# # COLAB ONLY
# renderer.render(box_size,
#                 {
#                     'particle': renderer.Sphere(system_positions, diameter=diameters)
#                 },
#                 resolution=(512, 512))

So our `system_` objects store the information about the system of `point_union` particles.

## Simulation of assembly

For now, I'll use Morse potentials with a defined depth `D0` for the patchy interactions, and will use the soft-sphere repulsion between the central particles (since that's more realistic in our case, I suppose).

In [ ]:
# ── Interaction matrices ──────────────────────────────────────────────────────
# Both potentials use 2×2 epsilon matrices indexed by (species_i, species_j).
# Species 0 = central site, species 1 = patch site.

# Morse epsilon: only the [1,1] entry is non-zero → patches attract patches.
# To add colored patches (e.g. only complementary patch types bind), expand this
# to a larger matrix with zeros for non-complementary species pairs.
morse_interaction_matrix = jnp.array([[D0]])
morse_eps = jnp.pad(morse_interaction_matrix, pad_width=(1, 0))  # → [[0,0],[0,D0]]

# LJ epsilon: only the [0,0] entry is non-zero → centers repel centers.
lj_eps = jnp.zeros((2, 2))
lj_eps = lj_eps.at[0, 0].set(1.0)

lj_sigma = float(RADIUS * 2.0)  # particle diameter used as LJ length scale

# ── Cutoff distances ──────────────────────────────────────────────────────────
# LJ uses a standard 2.5σ cutoff, which includes the full attractive well and
# provides enough range for particles to feel each other before contact.
# The capping at 0.45*box_size ensures the cutoff is < box/2 (required for PBC)
# even when testing with small N (small box).
lj_r_cutoff_abs = min(2.5 * lj_sigma, float(box_size) * 0.45)
lj_r_onset_abs  = 0.8 * lj_r_cutoff_abs  # smooth switch-off begins here
# lennard_jones_* functions scale r_onset/r_cutoff by sigma internally,
# so we pass dimensionless multiples here.
lj_r_cutoff = lj_r_cutoff_abs / lj_sigma
lj_r_onset  = lj_r_onset_abs  / lj_sigma

# Morse cutoff of 3.0 gives patches room to interact before coming into contact.
# morse_* functions do NOT scale by sigma, so these are absolute length units.
# With alpha=5.0 the well decays as exp(-5r), so 3.0 captures essentially all of it.
# To increase the interaction range (useful at low density), reduce alpha toward 1.0.
morse_r_cutoff = min(3.0, float(box_size) * 0.45)
morse_r_onset  = 0.85 * morse_r_cutoff  # must be < r_cutoff for correct smooth envelope

# ── Neighbor list ─────────────────────────────────────────────────────────────
# Rather than computing all N*(N-1)/2 pair interactions at every step, the neighbor
# list pre-computes which pairs are within the cutoff radius and only evaluates
# those.  For N=100 this gives ~100x fewer pairs than the dense calculation.
#
# The list is built on all 7*N flat-particle sites simultaneously (not just centres),
# so that both the center–center LJ and the patch–patch Morse share one list.
#
# dr_threshold: if any site moves more than this since the last rebuild,
#   the list is rebuilt automatically inside the scan.  Must be < (nl_r_cutoff - max_cutoff)/2.
# capacity_multiplier: pre-allocates this factor extra space so that rebuilds
#   are rare.  Increase to 2.0+ if you see overflow warnings.
nl_r_cutoff = lj_r_cutoff_abs  # LJ dominates; Morse cutoff (3.0) is always smaller

neighbor_fn = partition.neighbor_list(
    disp_fn, box_size, nl_r_cutoff,
    dr_threshold=0.5,
    capacity_multiplier=1.75,
    format=partition.OrderedSparse,  # sparse format is most memory-efficient for large N
)

# ── Energy functions ──────────────────────────────────────────────────────────
# We build both potentials directly via smap.pair_neighbor_list so they share the
# single neighbor list above.  (The convenience wrappers lennard_jones_neighbor_list /
# morse_neighbor_list each allocate their own list internally, which we avoid.)
#
# multiplicative_isotropic_cutoff wraps a potential with a C¹-smooth switch that
# goes from 1 (at r_onset) to 0 (at r_cutoff), keeping forces continuous.

# Lennard-Jones for center–center repulsion (+attraction at medium range).
# The full LJ has a minimum at r = 2^(1/6)*sigma ≈ 5.6, giving a modest
# cohesive force that helps clusters stay together between patch interactions.
lj_energy_fn = smap.pair_neighbor_list(
    energy.multiplicative_isotropic_cutoff(
        energy.lennard_jones,
        lj_r_onset_abs,
        lj_r_cutoff_abs,
    ),
    space.canonicalize_displacement_or_metric(disp_fn),
    ignore_unused_parameters=True,
    species=2,          # tells smap to use a 2×2 epsilon matrix
    sigma=lj_sigma,
    epsilon=lj_eps,
)

# Morse for patch–patch attraction.
# U(r) = D0*(1 − exp(−alpha*(r−sigma)))² − D0
# With sigma=0 the minimum is at contact (r=0) and the range is ~1/alpha ≈ 0.2.
# Increase D0 to make binding stronger; decrease alpha to make it longer-ranged.
morse_energy_fn = smap.pair_neighbor_list(
    energy.multiplicative_isotropic_cutoff(
        energy.morse,
        morse_r_onset,
        morse_r_cutoff,
    ),
    space.canonicalize_displacement_or_metric(disp_fn),
    ignore_unused_parameters=True,
    species=2,
    sigma=0.0,
    epsilon=morse_eps,
    alpha=5.0,
)

# Combined pair energy: called with flat positions (N*7, 3) and a neighbor list.
def pair_energy_fn(R, neighbor, **kwargs):
    return (
        lj_energy_fn(R, neighbor=neighbor, **kwargs)
        + morse_energy_fn(R, neighbor=neighbor, **kwargs)
    )

# point_energy_neighbor_list wraps pair_energy_fn so it accepts a RigidBody
# directly (converting to flat positions internally) and returns:
#   rb_neighbor_fn — neighbor list functions that accept/return RigidBody objects
#   energy_fn      — energy function that takes (RigidBody, NeighborList) → scalar
rb_neighbor_fn, energy_fn = rigid_body.point_energy_neighbor_list(
    pair_energy_fn, neighbor_fn, oct_shape
)

# Allocate the initial neighbor list and sanity-check the energy.
neighbors = rb_neighbor_fn.allocate(system_rigid_body)
print('Energy of the initial state: {:.4f}'.format(float(energy_fn(system_rigid_body, neighbors))))


In [ ]:
# ── Initialise the integrator ─────────────────────────────────────────────────
# nvt_langevin implements the BAOAB Langevin splitting, which correctly samples
# the NVT (constant temperature) ensemble.  The thermostat target temperature
# is passed at each step via the kT= kwarg, allowing annealing schedules.
# gamma must be a RigidBody with separate translational / rotational coefficients.
init_fn, step_fn = simulate.nvt_langevin(energy_fn, shift_fn, dt, kT(0.0), gamma=gamma)

# init_fn draws initial momenta from the Maxwell–Boltzmann distribution.
# neighbor= is passed so the first force evaluation uses the correct neighbor list.
state = init_fn(key, system_rigid_body, mass=oct_shape.mass(), neighbor=neighbors)

n_chunks = num_steps // save_every  # number of trajectory frames we will collect

# Capture the mass once outside the scan; it is constant and will be used
# inside the JIT to compute kinetic energies at each frame.
_mass = state.mass

# ── Main simulation loop (nested lax.scan) ────────────────────────────────────
# We use a nested lax.scan structure to keep everything on-device:
#
#   outer_step  runs every save_every steps and emits one frame + diagnostics.
#   inner_step  advances the simulation by one MD step each call:
#               1. Rebuild the neighbor list if any site has drifted > dr_threshold.
#               2. Advance positions and momenta with the Langevin step.
#
# This avoids Python-level loops and the device↔host syncs they require, giving
# the same performance as a single flat scan over all num_steps steps.
@jit
def simulate_all(state, neighbors):
    def outer_step(carry, _):
        state, nbrs = carry

        # Inner scan: run save_every Langevin steps, returning only final state.
        def inner_step(carry, _):
            state, nbrs = carry
            # Update neighbor list.  nbrs.update accepts RigidBody directly because
            # rb_neighbor_fn replaced the internal update_fn to handle the
            # RigidBody → flat-positions conversion automatically.
            nbrs = nbrs.update(state.position)
            new_state = step_fn(state, neighbor=nbrs, kT=kT(0.0))
            return (new_state, nbrs), None

        (new_state, new_nbrs), _ = lax.scan(
            inner_step, (state, nbrs), None, length=save_every
        )

        # Convert the current RigidBody state to Cartesian site positions for
        # storage.  vmap applies the per-body transform in parallel over N bodies;
        # reshape gives (N*7, 3): all sites in particle-index order.
        frame = vmap(rigid_body.transform, (0, None))(new_state.position, oct_shape).reshape(
            number_of_particles * 7, 3
        )

        # ── Per-frame diagnostics ─────────────────────────────────────────────
        # ke_total = translational KE + rotational KE.
        # rigid_body.kinetic_energy handles the quaternion-conjugate-momentum →
        # angular-momentum conversion before computing 0.5 * Σ L²/I.
        ke_total = rigid_body.kinetic_energy(new_state.position, new_state.momentum, _mass)
        # Translational KE = 0.5 * Σ_i |p_i|² / M_i  (sum over N rigid bodies)
        ke_trans = 0.5 * jnp.sum(new_state.momentum.center ** 2 / _mass.center)
        ke_rot   = ke_total - ke_trans
        # Temperature = 2*KE / (n_dof * k_B); JAX-MD uses k_B=1 (reduced units).
        # n_dof = 3N translational + 3N rotational = 6N for nonlinear rigid bodies.
        temp = rigid_body.temperature(new_state.position, new_state.momentum, _mass)

        return (new_state, new_nbrs), (frame, ke_trans, ke_rot, temp)

    (final_state, final_nbrs), (frames, ke_trans, ke_rot, temps) = lax.scan(
        outer_step, (state, neighbors), None, length=n_chunks
    )
    return final_state, final_nbrs, frames, ke_trans, ke_rot, temps

print('Running simulation (compiling on first run)...')
final_state, final_nbrs, frames, ke_trans_arr, ke_rot_arr, temp_arr = simulate_all(state, neighbors)
jax.block_until_ready(frames)

# Overflow means the neighbor list ran out of pre-allocated capacity.
# Fix: increase capacity_multiplier in the energy cell and re-run.
if final_nbrs.did_buffer_overflow:
    print('Warning: neighbor list overflowed. Increase capacity_multiplier in the energy cell.')

# Transfer all results from device (GPU/XLA) to host numpy arrays.
site_positions = np.array(frames)       # (n_chunks, N*7, 3)  — xyz for all sites
ke_trans_arr   = np.array(ke_trans_arr) # (n_chunks,)          — translational KE
ke_rot_arr     = np.array(ke_rot_arr)   # (n_chunks,)          — rotational KE
temp_arr       = np.array(temp_arr)     # (n_chunks,)          — instantaneous temperature
print(f'Done. shape: {site_positions.shape}')


In [ ]:
import matplotlib.pyplot as plt

# ── Temperature and kinetic energy diagnostics ────────────────────────────────
# Two-panel plot produced from the per-frame arrays collected during simulate_all.
#
# Panel 1 — Temperature vs time
#   Shows how quickly the thermostat equilibrates and whether it stays near target.
#   The dashed line marks the target kT value.
#   If temperature drifts far from target, try increasing gamma (stronger thermostat)
#   or reducing dt (more stable integration).
#
# Panel 2 — Kinetic energy vs time
#   Translational KE = 0.5 * Σ |p|² / M  (3N translational DOF → target = 1.5 N kT)
#   Rotational KE    = total KE − translational KE  (3N rotational DOF → target = 1.5 N kT)
#   Total KE         = translational + rotational     (target = 3 N kT)
#
#   It is normal for rotational KE to appear lower than translational KE in this
#   model: the patch masses are ~1e-8, giving a very small moment of inertia
#   (~2e-7).  The Langevin rotational relaxation time τ_rot = I/γ_rot is then
#   ~10⁻⁸, far shorter than dt = 10⁻³, so rotational DOF are overdamped and
#   do not thermalize correctly at this time step.

times = np.arange(n_chunks) * save_every * dt  # physical time for each saved frame

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

ax1.plot(times, temp_arr, color='tab:red', lw=0.8)
ax1.axhline(kT(0.0), color='k', ls='--', lw=1.0, label=f'target kT = {kT(0.0):.2f}')
ax1.set_ylabel('Temperature')
ax1.legend(fontsize=9)

ax2.plot(times, ke_trans_arr, label='translational', color='tab:blue',   lw=0.8)
ax2.plot(times, ke_rot_arr,   label='rotational',    color='tab:orange',  lw=0.8)
ax2.plot(times, ke_trans_arr + ke_rot_arr, label='total', color='tab:green', ls='--', lw=1.0)
ax2.set_ylabel('Kinetic energy')
ax2.set_xlabel('Time')
ax2.legend(fontsize=9)

fig.tight_layout()
plt.show()

In [98]:
site_positions[0].shape

(70, 3)

In [ ]:
# Write the trajectory to an extended-XYZ file readable by Ovito, VESTA, and ASE.
#
# Extended XYZ format (used by Ovito):
#   line 1:  total number of sites in this frame
#   line 2:  comment / column-layout declaration
#   lines 3…N+2: <type> <x> <y> <z> <radius>
#
# We use two particle type labels here:
#   "A" — central site of each octahedron  (large sphere, radius = RADIUS)
#   "E" — patch sites along ±x, ±y, ±z    (small sphere, radius = 0.5)
#
# To add patch coloring (e.g. axis-1 patches bind only axis-1 patches on neighbors),
# replace "E" with distinct letters per axis ("B", "C", "D") and update the
# interaction epsilon matrix so only matching types attract each other.
#
# To load in Ovito:
#   File → Load File → select trajectory.xyz
#   Ovito will auto-detect the extended XYZ format and the "radius" property.
#   Add "Assign Color" modifier and color by "Particle Type" to distinguish
#   central bodies from patches visually.
traj = site_positions  # shape (n_frames, N*7, 3)

n_frames, n_particles, _ = traj.shape

types = []
radii = []

for i in range(number_of_particles):
    types.append("A");     radii.append(RADIUS)  # central site
    types += ["E"] * 2;    radii += [0.5] * 2    # ±x patches
    types += ["E"] * 2;    radii += [0.5] * 2    # ±y patches
    types += ["E"] * 2;    radii += [0.5] * 2    # ±z patches

types = np.array(types)
radii = np.array(radii)

with open("trajectory.xyz", "w") as f:
    for t in range(n_frames):
        f.write(f"{n_particles}\n")
        # Properties line tells Ovito the column layout.
        f.write("Properties=species:S:1:pos:R:3:radius:R:1\n")

        for i in range(n_particles):
            x, y, z = traj[t, i]
            f.write(f"{types[i]} {x:.6f} {y:.6f} {z:.6f} {radii[i]}\n")

In [20]:
traj = site_positions  # shape (num_steps/save_every, 70, 3)

n_frames, n_particles, _ = traj.shape

# define particle types
types = []
radii = []

for i in range(number_of_particles):
    types.append("A")
    radii.append(2.5)        # central particle

    types += ["E"] * 2
    radii += [0.5] * 2       # axis 1 patches

    types += ["E"] * 2       # axis 2 patches
    radii += [0.5] * 2

    # types += ["P"] * 2
    types += ["E"] * 2
    radii += [0.5] * 2       # axis 3 patches

types = np.array(types)
radii = np.array(radii)

with open("trajectory.xyz", "w") as f:
    for t in range(n_frames):
        f.write(f"{n_particles}\n")
        f.write("Properties=species:S:1:pos:R:3:radius:R:1\n")

        for i in range(n_particles):
            x, y, z = traj[t, i]
            r = radii[i]
            f.write(f"{types[i]} {x} {y} {z} {r}\n")